<a href="https://colab.research.google.com/github/Higashi88jp/Matsuo_Lab/blob/main/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 基礎講座　最終課題: NYUv2 セマンティックセグメンテーション

## 概要
RGB画像から、画像内の各ピクセルがどのクラスに属するかを予測するセマンティックセグメンテーションタスク.

### データセット
- データセット: NYUv2 dataset
- 訓練データ: 795枚
- テストデータ: 654枚
- 入力: RGB画像 + 深度マップ（元画像サイズは可変）
- 出力: 13クラスのセグメンテーションマップ
- 評価指標: Mean IoU (Intersection over Union)

### データセットの詳細（[NYU Depth Dataset V2](https://cs.nyu.edu/~fergus/datasets/nyu_depth_v2.html)）
- 画像は屋内シーンを撮影したもので、家具や壁、床などの物体が含まれています.
- 各画像に対して13クラスのセグメンテーションラベルが提供されます.
- データは以下のディレクトリ構造で提供:
```
data/NYUv2/
├─train/
│  ├─image/      # RGB画像
│  │    000000.png
│  │    ...
│  │
│  ├─depth/      # 深度マップ
│  │    000000.png
│  │    ...
│  │
│  └─label/      # 13クラスセグメンテーション（教師ラベル）
│       000000.png
│       ...
└─test/
   ├─image/      # RGB画像
   │    000000.png
   │    ...
   │  ├─depth/   # 深度マップ
   │    000000.png
   │    ...
```

### タスクの詳細
- 入力のRGB画像と深度マップから、各ピクセルが13クラスのどれに属するかを予測するタスクです.
- 評価はMean IoUを使用します．
  - 各クラスごとにIoUを計算し、その平均を取ります.
  - IoUは以下の式で計算:
  $$IoU = \frac{TP}{TP + FP + FN}$$
    - TP: True Positive（正しく予測されたピクセル数）
    - FP: False Positive（誤って予測されたピクセル数）
    - FN: False Negative（見逃したピクセル数）

### 前処理
- 入力画像は512×512にリサイズされます.
- ピクセル値は0-1に正規化されます.
- セグメンテーションラベルは0-12の整数値（13クラス）です．
  - 255はignore index（評価から除外）

### 提出形式
- テスト画像（RGB + Depth）の各ピクセルに対してクラス（0~12）を予測したものをnumpy配列として保存されます.
- ファイル名: `submission.npy`
- 配列の形状: [テストデータ数, 高さ, 幅]
- 各ピクセルの値: 0-12の整数（予測クラス）



## 考えられる工夫の例
- 事前学習モデルの fine-tuning
    - ImageNetなどで事前学習されたモデルを本データセットでfine-tuningすることで性能向上が見込めます.
- 損失関数の再設計
    - クラスごとの出現頻度に応じて損失を補正するように損失関数を設計すると、クラス分布の不均衡に対してロバストな学習ができます.
- 画像の前処理
    - RandomResizedCrop / Flip / ColorJitter 等のデータ拡張を追加することで，汎化性能の向上が見込めます．

## 修了要件を満たす条件
- ベースラインでは，omnicampus 上での性能評価において， 36.4% となります．したがって，ベースラインである 36.4% を超えた提出のみ，修了要件として認めます．
- ベースラインから改善を加えることで， 50%以上に性能向上することを運営で確認しています．こちらを 1つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

### データの準備
データをダウンロードした際に，google drive したため，利用するために google drive をマウントする必要があります．また， drive 上で展開することができないため，/content ディレクトリ下にコピーし "data.zip" を展開します．  
google drive 上に "data.zip" が配置されていない場合は実行できません．google drive 上に "data.zip" (**831MB**) を配置することが可能であれば，"data_download.ipynb" を先に実行してください．難しい場合は，omnicampus 演習環境を利用してください．．



In [ ]:
# omnicampus 上では 4 セル目まで実行不要
# ドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# データダウンロード用の notebook にてgoogle drive への保存後，
# 反映に時間がかかる可能性がありますので，google drive のマウント後，
# data.zip がディレクトリ内にあることを確認してから実行してください．
# data.zip を /content 下にコピーする
# !cp "/content/drive/MyDrive/data.zip" "/content"
!cp "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/data.zip" "/content"

In [ ]:
# カレントディレクトリ下のファイル群を確認
# data.zip が表示されれば問題ないです
%ls

In [ ]:
# データを解凍する一度解凍しているので、再度解凍しないようにする
!unzip data.zip
!mkdir data
!mv train test data/

omnicampus 演習環境では，`gsutil`を利用してデータをダウンロードすることができます．ダウンロード後，data ディレクトリが存在するディレクトリをカレントディレクトリとするだけで良いです．



In [ ]:
# omnicampus 実行用
# 以下の例では/workspace/Segmentation/split_data_scripts/omnicampus に data ディレクトリがあると想定
# %cd /workspace/Segmentation/split_data_scripts_omnicampus
# %cd /content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/data

In [ ]:
# omnicampus 実行用
# !pip install numpy==1.22.2 h5py scikit-image

# import library

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import random
import time
from tqdm import tqdm
import numpy as np
from scipy.io import loadmat
from PIL import Image
import torch
import torch.nn as nn
from torch import optim
import torch.utils.data as data
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import VisionDataset
from torchvision import transforms
from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    RandomHorizontalFlip,
    ColorJitter,
    GaussianBlur,
    Resize,
    ToTensor,
    Normalize,
    Lambda,
    InterpolationMode
)
import torchvision.transforms.functional as F

class JointTransform:
    def __init__(self, image_size, hflip_prob=0.5):
        self.image_size = image_size
        self.hflip_prob = hflip_prob

        # ColorJitter
        self.color_jitter = transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        )

    def __call__(self, image, depth, label=None):
        # =========================
        # ✅ RandomResizedCrop（ここが追加）
        # =========================
        if random.random() < 0.8:
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                image,
                scale=(0.7, 1.0),
                ratio=(0.75, 1.33)
            )

            image = F.resized_crop(image, i, j, h, w, self.image_size, Image.BILINEAR)
            depth = F.resized_crop(depth, i, j, h, w, self.image_size, Image.BILINEAR)

            if label is not None:
                label = F.resized_crop(
                label, i, j, h, w, self.image_size, Image.NEAREST
            )

        else:
            # crop使わない場合
            image = image.resize(self.image_size, Image.BILINEAR)
            depth = depth.resize(self.image_size, Image.BILINEAR)

            if label is not None:
                label = label.resize(self.image_size, Image.NEAREST)

        # --- flip ---
        if random.random() < self.hflip_prob:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            depth = depth.transpose(Image.FLIP_LEFT_RIGHT)
            if label is not None:
                label = label.transpose(Image.FLIP_LEFT_RIGHT)

        # --- ColorJitter（RGBのみ） ---
        if random.random() < 0.8:
            image = self.color_jitter(image)

        # --- tensor化 ---
        image = transforms.ToTensor()(image)
        depth = transforms.ToTensor()(depth)

        if label is not None:
            label = torch.from_numpy(np.array(label)).long()
            return image, depth, label

        return image, depth

from torch.cuda.amp import autocast, GradScaler
from dataclasses import dataclass


# DataLoader

In [ ]:
# カラーマップ生成関数：セグメンテーションの可視化用
def colormap(N=256, normalized=False):
    def bitget(byteval, idx):
        return ((byteval & (1 << idx)) != 0)

    dtype = 'float32' if normalized else 'uint8'
    cmap = np.zeros((N, 3), dtype=dtype)
    for i in range(N):
        r = g = b = 0
        c = i
        for j in range(8):
            r = r | (bitget(c, 0) << 7-j)
            g = g | (bitget(c, 1) << 7-j)
            b = b | (bitget(c, 2) << 7-j)
            c = c >> 3

        cmap[i] = np.array([r, g, b])

    cmap = cmap/255 if normalized else cmap
    return cmap

# NYUv2データセット：RGB画像、セグメンテーション、深度、法線マップを提供するデータセット
class NYUv2(VisionDataset):
    """NYUv2 dataset

    Args:
        root (string): Root directory path.
        split (string, optional): 'train' for training set, and 'test' for test set. Default: 'train'.
        target_type (string, optional): Type of target to use, ``semantic``, ``depth``.
        transform (callable, optional): A function/transform that takes in an PIL image and returns a transformed version.
        target_transform (callable, optional): A function/transform that takes in the target and transforms it.
    """
    cmap = colormap()
    def __init__(self,
                 root,
                 split='train',
                 include_depth=False,
                 transform=None,
                 target_transform=None,
                 ):
        super(NYUv2, self).__init__(root, transform=transform, target_transform=target_transform)

        # データセットの基本設定
        assert(split in ('train', 'test'))
        self.root = root
        self.split = split
        self.include_depth = include_depth
        self.train_idx = np.array([255, ] + list(range(13)))  # 13クラス分類用

        # 画像ファイルのパスリストを作成
        img_names = os.listdir(os.path.join(self.root, self.split, 'image'))
        img_names.sort()
        images_dir = os.path.join(self.root, self.split, 'image')
        self.images = [os.path.join(images_dir, name) for name in img_names]

        label_dir = os.path.join(self.root, self.split, 'label')
        if (self.split == 'train'):
          self.labels = [os.path.join(label_dir, name) for name in img_names]
          self.targets = self.labels

        depth_dir = os.path.join(self.root, self.split, 'depth')
        self.depths = [os.path.join(depth_dir, name) for name in img_names]

    def __getitem__(self, idx):
        image = Image.open(self.images[idx])
        depth = Image.open(self.depths[idx])

        if self.split == 'test':
            if self.transform is not None:
                image, depth = self.transform(image, depth)
            return image, depth

        target = Image.open(self.targets[idx])

        if self.transform is not None:
            image, depth, target = self.transform(image, depth, target)

        return image, depth, target

    def __len__(self):
        return len(self.images)

# Model Section


In [ ]:
# 2つの畳み込み層とバッチ正規化、ReLUを含むブロック
# UNetの各層で使用される基本的な畳み込みブロック
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

# UNetモデル：エンコーダ・デコーダ構造のセグメンテーションモデル
class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()

        # ===== Encoder =====
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # ===== ⭐ Bottleneck（追加ポイント） =====
        self.bottleneck = DoubleConv(512, 1024)

        # ===== Decoder =====
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512 + 512, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256 + 256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128 + 128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64 + 64, 64)

        self.final = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder（skip connection）
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.final(d1)


# Train and Valid

In [ ]:
# config
@dataclass
class TrainingConfig:
    # データセットパス
    dataset_root: str = "data"

    # データ関連
    batch_size: int = 32
    num_workers: int = 4

    # モデル関連
    in_channels: int = 3
    num_classes: int = 13  # NYUv2データセットの場合

    # 学習関連
    epochs: int = 100
    learning_rate: float = 0.001
    weight_decay: float = 1e-4

    # データ分割関連
    train_val_split: float = 0.8  # 訓練データの割合

    # デバイス設定
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # チェックポイント関連
    checkpoint_dir: str = "checkpoints"
    save_interval: int = 5  # エポックごとのモデル保存間隔

    # データ拡張・前処理関連
    image_size: tuple = (256, 256)
    normalize_mean: tuple = (0.485, 0.456, 0.406)  # ImageNetの標準化パラメータ
    normalize_std: tuple = (0.229, 0.224, 0.225)

    def __post_init__(self):
        os.makedirs(self.checkpoint_dir, exist_ok=True)

In [ ]:
def set_seed(seed):
    """
    シードを固定する．

    Parameters
    ----------
    seed : int
        乱数生成に用いるシード値．
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
set_seed(42)
# 設定の初期化
config = TrainingConfig(
    dataset_root='/content/data',
    batch_size=16,
    num_workers=2,           # ← warning対策で 2 に下げる
    learning_rate=3e-4,
    epochs=60,               # ← まずは減らす
    image_size=(512, 512),   # ← 縦横を要確認
    in_channels=4  # RGB(3チャネル) + Depth(1チャネル)
)

'''
データセットのディレクトリ構造：
    data/NYUv2/
    ├─train/
    │  ├─image/      # RGB画像（入力）
    │  │    000000.png
    │  │    ...
    |  ├─depth/      # 深度画像（入力）
    |  │    000000.png
    |  │    ...
    │  └─label/      # 13クラスセグメンテーション（教師ラベル）
    │       000000.png
    │       ...
    └─test/
       ├─image/      # RGB画像（入力）
       │    000000.png
       │    ...
       ├─depth/      # 深度画像（入力）
       │    000000.png
       │    ...
'''

print("dataset_root exists:", os.path.exists(config.dataset_root))
print("train/image exists:", os.path.exists(os.path.join(config.dataset_root, 'train', 'image')))
print("train/depth exists:", os.path.exists(os.path.join(config.dataset_root, 'train', 'depth')))
print("train/label exists:", os.path.exists(os.path.join(config.dataset_root, 'train', 'label')))
print("test/image exists:", os.path.exists(os.path.join(config.dataset_root, 'test', 'image')))
print("test/depth exists:", os.path.exists(os.path.join(config.dataset_root, 'test', 'depth')))

# ------------------
#    Dataloader
# ------------------
# データセットの準備
# RGBデータセットとセグメンテーションラベルの読み込み
joint_train_transform = JointTransform(
    image_size=config.image_size,
    hflip_prob=0.5
)

joint_eval_transform = JointTransform(
    image_size=config.image_size,
    hflip_prob=0.0  # flipしない
)

train_dataset = NYUv2(
    root=config.dataset_root,
    split='train',
    include_depth=True,
    transform=joint_train_transform
)

test_dataset = NYUv2(
    root=config.dataset_root,
    split='test',
    include_depth=True,
    transform=joint_eval_transform
)

val_ratio = 0.1
val_size = int(len(train_dataset) * val_ratio)
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

'''
    train data:
        Type of batch: tuple
        Index 0 (入力データ):
            Type: torch.Tensor
            Shape: torch.Size([Batch, 3, N, M])
            Details: RGBテンソル
                    - チャネル0-2: RGB画像 (値域: 0-1)
        Index 1 (教師ラベル):
            Type: torch.Tensor
            Shape: torch.Size([Batch, N, M])
            Details: セグメンテーションマップ
                    - 値域: 0-12 (13クラス)
                    - 255: ignore index

    test data:
        Type of batch: torch.Tensor
        Shape: torch.Size([Batch, 3, N, M])
        Details: RGB画像 (値域: 0-1)
'''

# データローダーの作成
train_data = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
    persistent_workers=True
)

val_data = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
    persistent_workers=True
)

test_data = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
    persistent_workers=True
)

# ------------------
#    Device
# ------------------
device = config.device
print(f"Using device: {device}")
# RGB正規化用
rgb_mean = torch.tensor(config.normalize_mean, device=device).view(1, 3, 1, 1)
rgb_std  = torch.tensor(config.normalize_std,  device=device).view(1, 3, 1, 1)

# ------------------
#    Model
# ------------------
model = UNet(
    in_channels=config.in_channels,
    num_classes=config.num_classes
).to(device)

# ------------------
#    optimizer
# ------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.epochs
)

# ------------------
#    Metrics / Loss
# ------------------
def compute_miou(preds, labels, num_classes, ignore_index=255):
    preds = preds.view(-1)
    labels = labels.view(-1)

    mask = labels != ignore_index
    preds = preds[mask]
    labels = labels[mask]

    ious = []
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            continue
        ious.append(intersection / union)

    return sum(ious) / len(ious) if len(ious) > 0 else 0.0

class DiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=255, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)

        valid_mask = (targets != self.ignore_index)
        targets = targets.clone()
        targets[~valid_mask] = 0

        one_hot = torch.nn.functional.one_hot(targets, num_classes=self.num_classes)
        one_hot = one_hot.permute(0, 3, 1, 2).float()

        valid_mask = valid_mask.unsqueeze(1)
        probs = probs * valid_mask
        one_hot = one_hot * valid_mask

        intersection = (probs * one_hot).sum(dim=(0, 2, 3))
        union = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))

        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

class_weights = torch.tensor(
    [0.5, 0.7, 0.8, 1.0, 1.2, 1.2, 1.5, 1.5, 1.8, 1.8, 2.0, 2.0, 2.2],
    device=device
)
ce_loss = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)

dice_loss = DiceLoss(num_classes=config.num_classes, ignore_index=255)

# ------------------
#    Training
# ------------------
num_epochs = config.epochs
scaler = torch.amp.GradScaler('cuda')

best_miou = 0.0

for epoch in range(num_epochs):
    # ===== Train =====
    model.train()
    total_loss = 0.0
    print(f"on epoch: {epoch+1}")

    with tqdm(train_data) as pbar:
        for batch_idx, (image, depth, label) in enumerate(pbar):
            image = image.to(device, non_blocking=True).float()
            depth = depth.to(device, non_blocking=True).float()
            label = label.to(device, non_blocking=True)

            if label.ndim == 4 and label.size(1) == 1:
                label = label.squeeze(1)

            # RGBだけ正規化
            image = (image - rgb_mean) / rgb_std

            # depth が [B, H, W] の場合は [B, 1, H, W] にする
            if depth.ndim == 3:
                depth = depth.unsqueeze(1)

            # Depth を各サンプルごとに標準化
            depth = (depth - depth.mean(dim=(2, 3), keepdim=True)) / (
                depth.std(dim=(2, 3), keepdim=True) + 1e-6
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                x = torch.cat((image, depth), dim=1)
                pred = model(x)
                loss = ce_loss(pred, label) + 0.5 * dice_loss(pred, label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / len(train_data)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

    # ===== Validation =====
    model.eval()
    miou_total = 0.0
    count = 0

    with torch.no_grad():
        for image, depth, label in val_data:
            image = image.to(device, non_blocking=True).float()
            depth = depth.to(device, non_blocking=True).float()
            label = label.to(device, non_blocking=True)

            if label.ndim == 4 and label.size(1) == 1:
                label = label.squeeze(1)

            # RGBだけ正規化
            image = (image - rgb_mean) / rgb_std

            if depth.ndim == 3:
                depth = depth.unsqueeze(1)

            depth = (depth - depth.mean(dim=(2, 3), keepdim=True)) / (
                depth.std(dim=(2, 3), keepdim=True) + 1e-6
            )

            with torch.amp.autocast('cuda'):
                x = torch.cat((image, depth), dim=1)
                pred = model(x)

            pred = torch.argmax(pred, dim=1)

            miou = compute_miou(pred, label, config.num_classes)
            miou_total += miou
            count += 1

    mean_miou = miou_total / count if count > 0 else 0.0
    print(f"Epoch {epoch+1}, Mean IoU: {mean_miou:.4f}")

    if mean_miou > best_miou:
        best_miou = mean_miou
        torch.save(model.state_dict(), "best_model.pt")
        print(f"✅ Best updated: {best_miou:.4f}")

    scheduler.step()
    print(f"Current LR: {scheduler.get_last_lr()[0]:.6f}")

# モデルの保存
current_time = time.strftime("%Y%m%d%H%M%S")
model_path = f"model_{current_time}.pt"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")


In [ ]:
# ==============================
# Final submission generation
# ==============================

# ✔ 必ずベストモデルを使う
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

# ✔ 提出用に名前を統一（最重要）
torch.save(model.state_dict(), "model.pt")

predictions = []

with torch.no_grad():
    for image, depth in tqdm(test_data):
        image = image.to(device, non_blocking=True).float()
        depth = depth.to(device, non_blocking=True).float()

        # ✅ 学習時と同じ正規化
        image = (image - rgb_mean) / rgb_std

        # depth を [B,1,H,W] に
        if depth.ndim == 3:
            depth = depth.unsqueeze(1)

        # ✅ depth標準化（これ超重要）
        depth = (depth - depth.mean(dim=(2, 3), keepdim=True)) / (
            depth.std(dim=(2, 3), keepdim=True) + 1e-6
        )

        with torch.amp.autocast('cuda'):
            logits = model(torch.cat([image, depth], dim=1))

            # ======================
            # ✅ ここからTTA追加
            # ======================
            image_flip = torch.flip(image, dims=[3])  # 左右反転
            depth_flip = torch.flip(depth, dims=[3])

            logits_flip = model(torch.cat([image_flip, depth_flip], dim=1))

            # 推論結果を元に戻す
            logits_flip = torch.flip(logits_flip, dims=[3])

            # 平均
            logits = (logits + logits_flip) / 2
            # ======================

        pred = torch.argmax(logits, dim=1)
        predictions.append(pred.cpu().numpy())

predictions = np.concatenate(predictions, axis=0).astype(np.int64)

# ✅ 提出用ファイル
np.save("submission.npy", predictions)

print("✅ submission.npy saved:", predictions.shape, predictions.dtype)


In [ ]:
import numpy as np
pred = np.load("submission.npy")
print(np.unique(pred))


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (セグメンテーション)」から提出してください．

- `submission.npy`
- `model.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
!ls

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

model_path = "model.pt"

# notebook_path = "/content/drive/MyDrive/Colab Notebooks/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb"
notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb"

with ZipFile("submission.zip",
             mode="w",
             compression=ZIP_DEFLATED,
             compresslevel=9) as zf:

    # submission.npy
    if os.path.exists("submission.npy"):
        zf.write("submission.npy")
        print("✅ submission.npy added")
    else:
        print("❌ submission.npy が見つからない")

    # model
    if os.path.exists(model_path):
        zf.write(model_path)
        print("✅ model added:", model_path)
    else:
        print("❌ モデルが見つからない:", model_path)

    # notebook
    if os.path.exists(notebook_path):
        zf.write(notebook_path,
                 arcname="DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb")
        print("✅ notebook added")
    else:
        print("❌ notebookが見つからない")


In [ ]:
# from zipfile import ZipFile, ZIP_DEFLATED

# model_path = "model.pt"

# notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb"

# with ZipFile("submission.zip",
#             !ls "/content/drive/MyDrive/"
#             !ls "/content/drive/MyDrive/Colab Notebooks/"$0
#             mode="w",

# with ZipFile("submission.zip",
#             mode="w",
#             compression=ZIP_DEFLATED,
#             compresslevel=9) as zf:
#     zf.write("submission.npy")
#     zf.write(model_path)
#     zf.write(notebook_path,
# arcname="DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb")

In [ ]:
# できたファイルのダウンロード
from google.colab import files
files.download('submission.zip')


In [ ]:
# ==============================
# Google Drive に提出ファイルを保存
# ==============================

import os
import shutil
from google.colab import drive

# Driveマウント（初回のみ認証が出ます）
drive.mount('/content/drive')

# 保存先フォルダ（自由に名前変更OK）
save_dir = "/content/drive/MyDrive/segmentation_submit"

# フォルダ作成（なければ作る）
os.makedirs(save_dir, exist_ok=True)

print("保存先:", save_dir)

# ==============================
# 必須ファイルの確認＆保存
# ==============================

# submission.npy
if os.path.exists("submission.npy"):
    shutil.copy("submission.npy", os.path.join(save_dir, "submission.npy"))
    print("✅ submission.npy 保存完了")
else:
    print("❌ submission.npy が存在しません")

# model.pt
if os.path.exists("model.pt"):
    shutil.copy("model.pt", os.path.join(save_dir, "model.pt"))
    print("✅ model.pt 保存完了")
else:
    print("❌ model.pt が存在しません")

# ==============================
# ZIPも作成して保存（推奨）
# ==============================
zip_path = "submission.zip"

# ZIP作成（直下に2ファイル入れる）
!zip -j submission.zip submission.npy model.pt

# Driveにコピー
if os.path.exists(zip_path):
    shutil.copy(zip_path, os.path.join(save_dir, "submission.zip"))
    print("✅ submission.zip 保存完了")
else:
    print("❌ submission.zip 作成失敗")

print("\n🎉 すべてDriveに保存されました！")

In [ ]:

import os
print(os.getcwd())
